<a href="https://colab.research.google.com/github/aldo02032004/naufaldo.github.io/blob/main/Topic_Prabowo%20ke%20Vladivostok_top_author_sentimen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Install Dependency

In [22]:
import os, sys, shutil
from google.colab import userdata

GH_TOKEN = userdata.get('GH_TOKEN')
REPO_DIR = "/content/GREAT-Tools"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

get_ipython().system(f'git clone -q https://{GH_TOKEN}@github.com/azmkto/GREAT-Tools.git {REPO_DIR}')

for mod_name in list(sys.modules):
    if mod_name == "great" or mod_name.startswith("great."):
        del sys.modules[mod_name]

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

get_ipython().system('pip install -q -U google-genai pandas tqdm emoji requests openpyxl ftfy')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.1 MB/s eta 0:00:00


## 1. Import Library

In [23]:
import re
import json
import time
import hashlib
import requests
import pandas as pd
from io import BytesIO
from tqdm.auto import tqdm
from IPython.display import display
from google import genai
from google.genai import types
from google.colab import drive

from great.text import clean_for_bert, SLANG as LIBRARY_SLANG

try:
    import emoji
    HAS_EMOJI_LIB = True
except ImportError:
    HAS_EMOJI_LIB = False
    print("[WARN] library 'emoji' tidak ada -> emoji akan dibuang, bukan dikonversi jadi teks.")

tqdm.pandas()

print("great loaded from:", __import__("great").__file__)
print("Commit terakhir  :", os.popen(f"git -C {REPO_DIR} rev-parse --short HEAD").read().strip())
print("Library siap.")

great loaded from: /content/GREAT-Tools/great/__init__.py
Commit terakhir  : 815deea
Library siap.


## 2. Set API Key Anthropic

Key diketik lewat input tersembunyi (tidak ke-log di notebook), jadi aman
kalau notebook ini nanti di-upload ke GitHub.

Alternatif lebih nyaman: pakai fitur **Secrets** Colab (ikon kunci di sidebar
kiri), simpan sebagai `ANTHROPIC_API_KEY`, lalu ganti isi cell ini dengan:

```python
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
```


In [18]:
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
gemini_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
print("API key sudah di-set.")

API key sudah di-set.


## 3. Konfigurasi

Edit bagian ini kalau nama kolom, daftar tema, atau bobot ranking berubah.
Semua cell di bawah memakai variabel dari sini.


In [20]:
# --- Kolom sumber data ---
COL_NO = "No"
COL_TYPE = "Type"
COL_HEADLINE = "Headline"
COL_MENTIONS = "Mentions"
COL_DATE = "Date"
COL_LINK = "Link"
COL_MEDIA = "Media"
COL_SENTIMENT = "Sentiment"
COL_AUTHOR_ID = "Author"
COL_FOLLOWERS = "Followers"
COL_RETWEETED = "Retweeted"
COL_FAVOURITED = "Favourited"

EXPECTED_COLS = [COL_NO, COL_TYPE, COL_HEADLINE, COL_MENTIONS, COL_DATE, COL_LINK,
                 COL_MEDIA, COL_SENTIMENT, COL_AUTHOR_ID, COL_FOLLOWERS,
                 COL_RETWEETED, COL_FAVOURITED]

# --- Tema ---
THEMES = ["kunjungan_prabowo", "rusia_putin", "lainnya"]
THEME_DESCRIPTIONS = {
    "kunjungan_prabowo": (
        "kunjungan Prabowo ke Rusia, kunjungan kenegaraan Prabowo, lawatan Prabowo, agenda Prabowo di Rusia, "
        "delegasi Indonesia ke Rusia, Prabowo temui Putin, pertemuan bilateral Prabowo Putin, "
        "Prabowo di Vladivostok, Prabowo di Moskow, Prabowo hadiri Eastern Economic Forum, "
        "Prabowo EEF, Prabowo forum ekonomi Rusia, kunjungan presiden RI ke Rusia, "
        "kerja sama Indonesia Rusia, nota kesepahaman Indonesia Rusia, MoU Indonesia Rusia, "
        "investasi Rusia ke Indonesia, kunjungan balasan Prabowo, protokoler kunjungan Prabowo, "
        "rombongan menteri dampingi Prabowo, jadwal kunjungan Prabowo Rusia, "
        "hasil pertemuan Prabowo Putin, pernyataan bersama Indonesia Rusia, "
        "kunjungan kenegaraan ke Kremlin, Prabowo di Kremlin, sambutan kenegaraan Prabowo Rusia, "
        "kereta kepresidenan Rusia, upacara penyambutan Prabowo, kunjungan luar negeri Prabowo, "
        "diplomasi Prabowo Rusia, agenda strategis Indonesia Rusia, "
        "lawatan kenegaraan Presiden Prabowo, kunjungan resmi Presiden Prabowo ke Rusia, "
        "Prabowo Subianto ke Rusia, Prabowo terbang ke Rusia, keberangkatan Prabowo ke Rusia, "
        "Prabowo tiba di Rusia, Prabowo pulang dari Rusia, kepulangan Prabowo dari Rusia, "
        "menteri luar negeri dampingi Prabowo, Sugiono dampingi Prabowo, Menlu RI ke Rusia, "
        "menteri pertahanan dampingi Prabowo, delegasi bisnis Indonesia Rusia, "
        "pengusaha Indonesia ikut kunjungan Rusia, pebisnis dampingi Prabowo, "
        "kesepakatan dagang Indonesia Rusia, perjanjian kerja sama Indonesia Rusia, "
        "kontrak dagang Indonesia Rusia, ekspor impor Indonesia Rusia, "
        "kerja sama pertahanan Indonesia Rusia, alutsista Rusia untuk Indonesia, "
        "kerja sama energi Indonesia Rusia, kerja sama nuklir Indonesia Rusia, "
        "PLTN Rusia Indonesia, Rosatom Indonesia, kerja sama pangan Indonesia Rusia, "
        "kunjungan Prabowo pasca kunjungan ke China, kunjungan Prabowo pasca KTT, "
        "reaksi publik kunjungan Prabowo Rusia, kritik kunjungan Prabowo ke Rusia, "
        "pujian kunjungan Prabowo ke Rusia, kontroversi kunjungan Prabowo Rusia, "
        "netralitas Indonesia kunjungan Rusia, politik luar negeri bebas aktif Prabowo, "
        "sikap Barat soal kunjungan Prabowo Rusia, respons AS soal kunjungan Prabowo Rusia, "
        "istana kepresidenan soal kunjungan Rusia, juru bicara presiden soal kunjungan Rusia, "
        "foto kunjungan Prabowo Rusia, video kunjungan Prabowo Rusia, momen Prabowo di Rusia, "
        "red carpet Prabowo Rusia, karpet merah Prabowo Rusia, penyambutan militer Prabowo Rusia"
    ),
    "rusia_putin": (
        "Rusia, Vladimir Putin, Presiden Rusia, Kremlin, Vladivostok, Moskow, Rusia Timur Jauh, "
        "Eastern Economic Forum, EEF Vladivostok, forum ekonomi Rusia, kebijakan luar negeri Rusia, "
        "hubungan Rusia dengan negara lain, sanksi terhadap Rusia, sanksi Barat ke Rusia, "
        "ekonomi Rusia, perdagangan Rusia, energi Rusia, gas Rusia, minyak Rusia, "
        "militer Rusia, angkatan bersenjata Rusia, perang Rusia Ukraina, konflik Rusia Ukraina, "
        "geopolitik Rusia, pernyataan Putin, pidato Putin, kebijakan Putin, "
        "Kementerian Luar Negeri Rusia, duta besar Rusia, kedutaan Rusia, "
        "kerja sama BRICS, Rusia BRICS, aliansi Rusia, mitra strategis Rusia, "
        "wilayah Timur Jauh Rusia, pelabuhan Vladivostok, industri Rusia, "
        "hubungan diplomatik dengan Rusia, kunjungan pejabat asing ke Rusia, "
        "Kremlin Moskow, Lapangan Merah, Istana Kremlin, juru bicara Kremlin, Dmitry Peskov, "
        "Sergey Lavrov, Menlu Rusia, diplomasi Rusia, Rusia dan negara Asia, Rusia dan ASEAN, "
        "Rusia dan Asia Tenggara, kunjungan kepala negara ke Rusia, tamu negara Rusia, "
        "ekonomi Rusia pasca sanksi, dampak sanksi terhadap Rusia, Rusia dan China, "
        "Rusia dan India, Rusia di panggung internasional, isolasi Rusia, Rusia G20"
    ),
    "lainnya": "topik di luar dua kategori di atas",
}

# --- Model & parameter LLM ---
GEMINI_MODEL = "gemini-3.1-flash-lite"
SLEEP_BETWEEN_CALLS = 4
GEN_CONFIG_DETERMINISTIC = dict(temperature=0.0, top_p=1.0, top_k=1)
THEME_CONFIDENCE_THRESHOLD = 0.6

# --- Bobot skor ranking top author (harus berjumlah 1.0) ---
WEIGHT_POST_COUNT = 0.6
WEIGHT_ENGAGEMENT = 0.25
WEIGHT_FOLLOWERS = 0.15

BATCH_SIZE_CLASSIFY = 40
TOP_N_AUTHORS_PER_THEME = 10
MAX_POSTS_PER_AUTHOR_SUMMARY = 15

CHECKPOINT_PATH = "/content/drive/MyDrive/df_clean_checkpoint.pkl"

# --- Normalisasi entity — Gemini sering kasih variasi penulisan ---
ENTITY_ALIASES = {
    "wowo": "Prabowo", "pak wowo": "Prabowo", "pak prabowo": "Prabowo",
    "prabowo subianto": "Prabowo", "psubianto": "Prabowo",
    "vlad putin": "Vladimir Putin", "putin": "Vladimir Putin", "vladimir putin": "Vladimir Putin",
    "vladivostock": "Vladivostok", "fefu": "Far Eastern Federal University",
}

def canonicalize_entity(name: str) -> str:
    key = name.strip().lower()
    return ENTITY_ALIASES.get(key, name.strip())

def canonicalize_list(names):
    return [canonicalize_entity(n) for n in names]

# --- Keyword prefilter (satu-satunya definisi) ---
KW_PRABOWO = [
    "prabowo", "presiden ri", "presiden indonesia", "presiden prabowo",
    "psubianto", "pak prabowo", "prabowosubianto", "#prabowo",
    "presiden subianto", "kepala negara ri", "kepala negara indonesia",
    "wowo", "pak wowo",
]
KW_RUSIA_KONTEKS = [
    "rusia", "russia", "putin", "vladimir putin", "kremlin", "moskow", "moscow",
    "vladivostok", "vladivostock", "eastern economic forum", "eef2025", "eef 2025",
    " eef ", "#eef", "rosatom", "lavrov", "peskov", "#rusia", "#putin", "#vladivostok",
]
KW_RUSIA_PUTIN_UMUM = KW_RUSIA_KONTEKS + ["ukraina", "brics", "sanksi rusia", "sanksi barat", "#brics"]
KW_DOMESTIK_EXCLUDE = [
    "gibran", "apbn", "elektabilitas", "makan bergizi", "mbg",
    "pilkada", "pemilu", "kades", "psi", "gerindra", "kpk", "kdm",
    "muktamar", "ormas", "grib", "habib rizieq", "islam ala prabowo",
    "desil", "kemensos", "baznas", "mui", "salat jumat", "ramadhan",
    "papua", "kalimantan", "jawa", "listrik", "pln",
]

def _contains_any(text_lower, kw_list):
    return any(kw in text_lower for kw in kw_list)

def keyword_match_theme(text):
    """Prefilter cepat, TIDAK dikirim ke LLM. Return tema atau None kalau ambigu."""
    text_lower = str(text).lower()
    ada_prabowo = _contains_any(text_lower, KW_PRABOWO)
    ada_rusia_konteks = _contains_any(text_lower, KW_RUSIA_KONTEKS)
    ada_rusia_umum = _contains_any(text_lower, KW_RUSIA_PUTIN_UMUM)

    if ada_prabowo and ada_rusia_konteks:
        return "kunjungan_prabowo"
    if not ada_prabowo and ada_rusia_umum:
        return "rusia_putin"
    if ada_prabowo and not ada_rusia_konteks and _contains_any(text_lower, KW_DOMESTIK_EXCLUDE):
        return "lainnya"
    if not ada_prabowo and not ada_rusia_umum:
        return "lainnya"
    return None

print("Konfigurasi siap.")

Konfigurasi siap.


## Step 1 — Load Data dari Google Sheets

Output: tabel mentah + daftar kolom, buat konfirmasi data ke-load dengan benar.


In [21]:
def resolve_xlsx_url(source: str) -> str:
    """Ubah link Google Sheets/Drive jadi url download .xlsx."""
    sheet_match = re.search(r"docs\.google\.com/spreadsheets/d/([a-zA-Z0-9-_]+)", source)
    drive_match = re.search(r"drive\.google\.com/file/d/([a-zA-Z0-9-_]+)", source)
    if sheet_match:
        return f"https://docs.google.com/spreadsheets/d/{sheet_match.group(1)}/export?format=xlsx"
    if drive_match:
        return f"https://drive.google.com/uc?export=download&id={drive_match.group(1)}"
    return source

def _download_bytes(url: str) -> BytesIO:
    session = requests.Session()
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = session.get(url, headers=headers, allow_redirects=True)
    resp.raise_for_status()
    content_type = resp.headers.get("Content-Type", "")
    if "html" in content_type.lower():
        raise ValueError(
            "Gagal download file asli (dapat HTML, bukan xlsx). "
            "Pastikan akses file/sheet sudah 'Anyone with the link'."
        )
    return BytesIO(resp.content)

def _find_best_sheet(xls: pd.ExcelFile, expected_cols: list, skiprows: int):
    best_sheet, best_match = xls.sheet_names[0], -1
    for name in xls.sheet_names:
        try:
            preview = pd.read_excel(xls, sheet_name=name, skiprows=skiprows, nrows=0)
            preview.columns = [c.strip() for c in preview.columns]
            match_count = sum(c in preview.columns for c in expected_cols)
            if match_count > best_match:
                best_match, best_sheet = match_count, name
        except Exception:
            continue
    return best_sheet

def load_raw_data(source: str, sheet_name=None, skiprows: int = 1) -> pd.DataFrame:
    url = resolve_xlsx_url(source)
    file_obj = _download_bytes(url) if url.startswith("http") else url
    xls = pd.ExcelFile(file_obj)

    if sheet_name is None:
        sheet_name = _find_best_sheet(xls, EXPECTED_COLS, skiprows)
        print(f"[INFO] Menggunakan sheet: '{sheet_name}'")

    df = pd.read_excel(xls, sheet_name=sheet_name, skiprows=skiprows)
    df.columns = [c.strip() for c in df.columns]

    missing = [c for c in EXPECTED_COLS if c not in df.columns]
    if missing:
        print(f"[WARNING] Kolom hilang: {missing}")

    available_cols = [c for c in EXPECTED_COLS if c in df.columns]
    return df[available_cols]

SOURCE = "https://docs.google.com/spreadsheets/d/1RWje0fuPtl1jHVzibr480gWfK0ivJkbY/edit?usp=sharing&ouid=116825097454650626545&rtpof=true&sd=true"

df_raw = load_raw_data(SOURCE)
print(f"Data berhasil dimuat: {df_raw.shape}")
display(df_raw.head())

[INFO] Menggunakan sheet: 'Sheet1'
Data berhasil dimuat: (66406, 12)


,No,Type,Headline,Mentions,Date,Link,Media,Sentiment,Author,Followers,Retweeted,Favourited
0,1,mention,NaN,Prabowo: Karhutla Jangan Sampai ke Zona Inti I...,2026-09-07 17:40:17,https://twitter.com/web/statuses/2096911425180...,Twitter,Positive,@kompascom,8042391,0,0
1,2,mention,NaN,Presiden Prabowo Subianto memberikan instruksi...,2026-09-07 17:33:56,https://twitter.com/web/statuses/2096909830162...,Twitter,Positive,@makcrigis_,177,0,0
2,3,mention,Prabowo Tegas Larang Segala Bentuk Pembakaran ...,Jakarta: Presiden RI Prabowo Subianto menginst...,2026-09-07 17:33:21,https://www.metrotvnews.com/read/b2lC62aV-prab...,News,Neutral,www.metrotvnews.com,0,0,0
3,4,mention,Prabowo Tegas Larang Segala Bentuk Pembakaran ...,Jakarta: Presiden RI Prabowo Subianto menginst...,2026-09-07 17:33:21,https://www.metrotvnews.com/read/b2lC62aV-prab...,News,Positive,www.metrotvnews.com,0,0,0
4,5,mention,NaN,Karhutla Kepulauan Meranti Tembus 133 Hektare ...,2026-09-07 17:33:20,https://twitter.com/web/statuses/2096909679599...,Twitter,Positive,@bukamata18,156,0,0


# Step 2 - FILTER AKUN MEDIA (satu-satunya definisi + eksekusi)


In [24]:
MEDIA_TYPE_ALWAYS = {"news"}
DOMAIN_SUFFIXES = (".com", ".id", ".co", ".net", ".org")
DOMAIN_TEXT_SUFFIXES = ("dotcom", "dotid", "dotco", "dotnet", "dotorg")
MEDIA_ACCOUNTS = {
    "detikcom", "kompascom", "cnnindonesia", "tempodotco", "antaranews",
    "cnbcindonesia", "liputan6dotcom", "liputan6", "sctv", "tvonenews", "kumparan",
    "beritasatu", "metrotvnews", "tribunnews", "republikaonline", "suaradotcom",
    "jawapos", "bbcindonesia", "voaindonesia", "narasitv", "sindonews",
    "vivacoid", "bisniscom", "hariankompas", "tirtoid", "alineadotid",
    "inilahcom", "merdekadotcom", "okezone", "idntimes", "medcomid",
    "rri", "tvri", "jpnndotcom", "grid", "inewsdotid", "fajar",
    "pikiranrakyat", "gatra", "nuonline", "radioelshinta", "mediaindonesia",
    "setkabgoid", "awani", "bharianmy", "bernamadotcom", "utusandotcom", "sinarharian",
    "malaysiakini", "thestar", "nst", "theedgemarkets",
    "rt", "actualidadrt", "rtcom", "sputnik", "sputniknews", "sputnikindonesia",
    "tass", "tassagency", "rianovosti", "ria", "kremlin", "kremlinru",
    "telesur", "telesurtv",
}
MEDIA_KEYWORDS_SUBSTRING = [
    "news", "media", "redaksi", "newsroom", "official", "humas", "koran",
    "awani", "gazette", "tv", "radio", "pers",
]
MEDIA_KEYWORDS_EXACT = []

def normalize_account(text: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(text).strip().lower())

def contains_media_keyword(author_lower: str) -> bool:
    if any(kw in author_lower for kw in MEDIA_KEYWORDS_SUBSTRING):
        return True
    tokens = re.split(r"[^a-z]+", author_lower)
    return any(tok in MEDIA_KEYWORDS_EXACT for tok in tokens if tok)

def is_media_account(row) -> bool:
    media_val = str(row.get(COL_MEDIA, "")).strip().lower()
    author_raw = str(row.get(COL_AUTHOR_ID, "")).strip().lower().lstrip("@")
    author_norm = normalize_account(author_raw)
    if media_val in MEDIA_TYPE_ALWAYS:
        return True
    if author_norm in MEDIA_ACCOUNTS:
        return True
    if contains_media_keyword(author_raw):
        return True
    if author_raw.endswith(DOMAIN_SUFFIXES):
        return True
    if author_norm.endswith(DOMAIN_TEXT_SUFFIXES):
        return True
    return False

df_raw[COL_FOLLOWERS] = pd.to_numeric(df_raw.get(COL_FOLLOWERS, 0), errors="coerce").fillna(0)
df_raw["is_media"] = df_raw.apply(is_media_account, axis=1)

n_media = df_raw["is_media"].sum()
n_nonmedia = (~df_raw["is_media"]).sum()
print(f"[INFO] {n_media} baris media dibuang, {n_nonmedia} baris non-media lanjut")

display(
    df_raw[[COL_AUTHOR_ID, COL_MEDIA, "is_media"]]
    .drop_duplicates(subset=COL_AUTHOR_ID)
    .head(20)
)

df_raw = df_raw[~df_raw["is_media"]].drop(columns=["is_media"]).reset_index(drop=True)

[INFO] 20420 baris media dibuang, 45986 baris non-media lanjut


,Author,Media,is_media
0,@kompascom,Twitter,True
1,@makcrigis_,Twitter,False
2,www.metrotvnews.com,News,True
4,@bukamata18,Twitter,False
5,@KolektorBatu,Twitter,False
6,news.detik.com,News,True
7,www.liputan6.com,News,True
9,Tribun MedanTV,Youtube,True
10,riauterkini.com,News,True
14,20.detik.com,News,True


# Step 3 - CLEANING TEKS

In [28]:
KAMUS_ALAY_TAMBAHAN = {
    "wowo": "prabowo", "pak": "bapak", "bu": "ibu", "min": "admin",
    "anjay": "sangat", "anjir": "sangat", "mantul": "bagus",
}
MERGED_SLANG = {**LIBRARY_SLANG, **KAMUS_ALAY_TAMBAHAN}

import great.text as _gt
_gt.SLANG = MERGED_SLANG
_gt.SLANG_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(k) for k in sorted(MERGED_SLANG.keys(), key=len, reverse=True)) + r")\b"
)

RE_RT_PREFIX = re.compile(r"^\s*RT\s*@[A-Za-z0-9_]+\s*:\s*", flags=re.IGNORECASE)
RE_ELONGATION = re.compile(r"(.)\1{2,}")
RE_MULTI_SPACE = re.compile(r"\s+")

def fix_elongation(text: str) -> str:
    return RE_ELONGATION.sub(r"\1\1", text)

def demojize_or_strip(text: str) -> str:
    if HAS_EMOJI_LIB:
        text = emoji.demojize(text, language="id" if "id" in emoji.LANGUAGES else "en")
        return text.replace("_", " ").replace(":", " ")
    return text

def build_raw_text(headline, mentions) -> str:
    headline = "" if pd.isna(headline) else str(headline).strip()
    mentions = "" if pd.isna(mentions) else str(mentions).strip()
    if not headline or headline.lower() == mentions.lower() or headline.lower() == "nan":
        return mentions or headline
    if not mentions:
        return headline
    return f"{headline}. {mentions}"

def clean_text(raw: str) -> str:
    """Gap yang gak dihandle clean_for_bert(): RT-prefix, emoji, elongasi.
    Sisanya (URL, mention, hashtag, slang) diserahin ke library."""
    if not isinstance(raw, str) or not raw.strip():
        return ""
    text = RE_RT_PREFIX.sub("", raw)
    text = demojize_or_strip(text)
    text = fix_elongation(text)
    text = clean_for_bert(text)
    return RE_MULTI_SPACE.sub(" ", text).strip()

def make_dedup_key(text_clean: str) -> str:
    key = re.sub(r"[^\w\s]", "", text_clean.lower())
    key = RE_MULTI_SPACE.sub(" ", key).strip()
    return hashlib.md5(key.encode("utf-8")).hexdigest()

headline_col = df_raw[COL_HEADLINE] if COL_HEADLINE in df_raw.columns else pd.Series([""] * len(df_raw))
mentions_col = df_raw[COL_MENTIONS] if COL_MENTIONS in df_raw.columns else pd.Series([""] * len(df_raw))
df_raw["text_raw_combined"] = [build_raw_text(h, m) for h, m in zip(headline_col, mentions_col)]

df_raw["text_clean"] = df_raw["text_raw_combined"].astype(str).progress_apply(clean_text)

before = len(df_raw)
df_clean = df_raw[df_raw["text_clean"].str.split().str.len().fillna(0) >= 3].copy()
print(f"[INFO] Buang {before - len(df_clean)} baris kosong/terlalu pendek")

df_clean["dedup_key"] = df_clean["text_clean"].apply(make_dedup_key)
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset="dedup_key", keep="first").drop(columns=["dedup_key"])
df_clean = df_clean.reset_index(drop=True)
print(f"[INFO] Buang {before - len(df_clean)} duplikat/retweet identik")
print(f"[OK] {len(df_clean)} baris tersisa setelah cleaning")

display(df_clean[["text_raw_combined", "text_clean"]].head(5))

  0%|          | 0/45986 [00:00<?, ?it/s]

[INFO] Buang 1496 baris kosong/terlalu pendek
[INFO] Buang 26905 duplikat/retweet identik
[OK] 17585 baris tersisa setelah cleaning


,text_raw_combined,text_clean
0,Presiden Prabowo Subianto memberikan instruksi...,Presiden Prabowo Subianto memberikan instruksi...
1,Karhutla Kepulauan Meranti Tembus 133 Hektare ...,Karhutla Kepulauan Meranti Tembus 133 Hektare ...
2,Presiden Prabowo mengatakan kapal induk dr Ita...,Presiden Prabowo mengatakan kapal induk dari I...
3,"Erupsi anak Krakatau, Presiden Prabowo perinta...","Erupsi anak Krakatau, Presiden Prabowo perinta..."
4,RT Kapolri Listyo Sigit Prabowo sambut kedatan...,RT Kapolri Listyo Sigit Prabowo sambut kedatan...


In [25]:
import re

MEDIA_TYPE_ALWAYS = {"news"}
DOMAIN_SUFFIXES = (".com", ".id", ".co", ".net", ".org")
DOMAIN_TEXT_SUFFIXES = ("dotcom", "dotid", "dotco", "dotnet", "dotorg")

MEDIA_ACCOUNTS = {
    "detikcom", "kompascom", "cnnindonesia", "tempodotco", "antaranews",
    "cnbcindonesia", "liputan6dotcom", "liputan6", "sctv", "tvonenews", "kumparan",
    "beritasatu", "metrotvnews", "tribunnews", "republikaonline", "suaradotcom",
    "jawapos", "bbcindonesia", "voaindonesia", "narasitv", "sindonews",
    "vivacoid", "bisniscom", "hariankompas", "tirtoid", "alineadotid",
    "inilahcom", "merdekadotcom", "okezone", "idntimes", "medcomid",
    "rri", "tvri", "jpnndotcom", "grid", "inewsdotid", "fajar",
    "pikiranrakyat", "gatra", "nuonline", "radioelshinta", "mediaindonesia",
    "setkabgoid",
    "awani", "bharianmy", "bernamadotcom", "utusandotcom", "sinarharian",
    "malaysiakini", "thestar", "nst", "theedgemarkets",
    "rt", "actualidadrt", "rtcom", "sputnik", "sputniknews", "sputnikindonesia",
    "tass", "tassagency", "rianovosti", "ria", "kremlin", "kremlinru",
    "telesur", "telesurtv",
}
MEDIA_KEYWORDS_SUBSTRING = [
    "news", "media", "redaksi", "newsroom", "official", "humas", "koran",
    "awani", "gazette", "tv", "radio", "pers",
]
MEDIA_KEYWORDS_EXACT = []


def normalize_account(text: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(text).strip().lower())


def contains_media_keyword(author_lower: str) -> bool:
    if any(kw in author_lower for kw in MEDIA_KEYWORDS_SUBSTRING):
        return True
    tokens = re.split(r"[^a-z]+", author_lower)
    return any(tok in MEDIA_KEYWORDS_EXACT for tok in tokens if tok)


def is_media_account(row) -> bool:
    media_val = str(row.get(COL_MEDIA, "")).strip().lower()
    author_raw = str(row.get(COL_AUTHOR_ID, "")).strip().lower().lstrip("@")
    author_norm = normalize_account(author_raw)

    if media_val in MEDIA_TYPE_ALWAYS:
        return True
    if author_norm in MEDIA_ACCOUNTS:
        return True
    if contains_media_keyword(author_raw):
        return True
    if author_raw.endswith(DOMAIN_SUFFIXES):
        return True
    if author_norm.endswith(DOMAIN_TEXT_SUFFIXES):
        return True
    return False


df_raw[COL_FOLLOWERS] = pd.to_numeric(df_raw.get(COL_FOLLOWERS, 0), errors="coerce").fillna(0)
df_raw_original = df_raw.copy()
df_raw["is_media"] = df_raw.apply(is_media_account, axis=1)

n_media = df_raw["is_media"].sum()
n_nonmedia = (~df_raw["is_media"]).sum()
print(f"[INFO] {n_media} baris media (dibuang SEBELUM cleaning)")
print(f"[INFO] {n_nonmedia} baris non-media (lanjut ke cleaning)")

print("\nContoh deteksi per akun unik:")
display(
    df_raw[[COL_AUTHOR_ID, COL_MEDIA, "is_media"]]
    .drop_duplicates(subset=COL_AUTHOR_ID)
    .head(20)
)

df_raw = df_raw[~df_raw["is_media"]].drop(columns=["is_media"]).reset_index(drop=True)
print(f"\n[INFO] df_raw sekarang berisi {len(df_raw)} baris non-media, siap dibersihkan")

[INFO] 0 baris media (dibuang SEBELUM cleaning)
[INFO] 45986 baris non-media (lanjut ke cleaning)

Contoh deteksi per akun unik:


,Author,Media,is_media
0,@makcrigis_,Twitter,False
1,@bukamata18,Twitter,False
2,@KolektorBatu,Twitter,False
3,@PolinaAntole,Twitter,False
4,@sarpur_889,Twitter,False
5,@mr4hm4n,Twitter,False
6,Eclip Podcast,Youtube,False
7,@redcize,Twitter,False
8,@orionnwashere,Twitter,False
9,@muffiniube,Twitter,False



[INFO] df_raw sekarang berisi 45986 baris non-media, siap dibersihkan


# Step 4 - KLASIFIKASI TEMA + NER — SATU LLM CALL PER BATCH (hemat kuota)

Gabung tema + lokasi + instansi + tokoh jadi 1 prompt, 1 jalur, tanpa duplikasi.

In [29]:
THEME_LIST_STR = "\n".join(f"- {t}: {THEME_DESCRIPTIONS[t]}" for t in THEMES)

FEWSHOT_COMBINED_EXAMPLES = """
Teks: "Presiden Prabowo tiba di Vladivostok untuk menghadiri Eastern Economic Forum dan bertemu Putin"
Jawaban: {"themes": ["kunjungan_prabowo", "rusia_putin"], "primary_theme": "kunjungan_prabowo", "confidence": 0.95, "lokasi": ["Vladivostok"], "instansi": ["Eastern Economic Forum"], "tokoh": ["Prabowo", "Putin"]}

Teks: "Putin sampaikan pidato di forum ekonomi Rusia soal kerja sama energi dengan negara Asia"
Jawaban: {"themes": ["rusia_putin"], "primary_theme": "rusia_putin", "confidence": 0.9, "lokasi": [], "instansi": [], "tokoh": ["Putin"]}

Teks: "Harga cabai naik jelang lebaran, pedagang mengeluh"
Jawaban: {"themes": ["lainnya"], "primary_theme": "lainnya", "confidence": 0.95, "lokasi": [], "instansi": [], "tokoh": []}

Teks: "wowo ke vladivostok ketemu putin, katanya mau bahas kerja sama militer"
Jawaban: {"themes": ["kunjungan_prabowo", "rusia_putin"], "primary_theme": "kunjungan_prabowo", "confidence": 0.85, "lokasi": ["Vladivostok"], "instansi": [], "tokoh": ["Prabowo", "Putin"]}

Teks: "Menlu Sugiono dampingi Prabowo dalam kunjungan ke Vladivostok temui Presiden Putin"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.9, "lokasi": ["Vladivostok"], "instansi": [], "tokoh": ["Sugiono", "Prabowo", "Putin"]}

Teks: "trafik motor macet parah di sekitar Kremlin depok gara-gara ada demo warga"
Jawaban: {"themes": ["lainnya"], "primary_theme": "lainnya", "confidence": 0.85, "lokasi": [], "instansi": [], "tokoh": []}

Teks: "Menko Airlangga, Menlu Sugiono, dan Menteri ESDM Bahlil ikut dampingi Prabowo terbang ke Vladivostok"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.9, "lokasi": ["Vladivostok"], "instansi": [], "tokoh": ["Airlangga Hartarto", "Sugiono", "Bahlil Lahadalia", "Prabowo"]}

Teks: "Rombongan terbatas Prabowo ke EEF diisi Rosan Roeslani, Brian Yuliarto, dan Seskab Teddy Indra Wijaya"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.88, "lokasi": [], "instansi": ["EEF"], "tokoh": ["Prabowo", "Rosan Roeslani", "Brian Yuliarto", "Teddy Indra Wijaya"]}

Teks: "Prabowo lepas landas dari Halim Perdanakusuma malam ini menuju Vladivostok"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.9, "lokasi": ["Halim Perdanakusuma", "Vladivostok"], "instansi": [], "tokoh": ["Prabowo"]}

Teks: "moderator sesi pleno EEF kirill tokarev tanya soal konektivitas maritim ke prabowo"
Jawaban: {"themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo", "confidence": 0.8, "lokasi": [], "instansi": ["EEF"], "tokoh": ["Kirill Tokarev", "Prabowo"]}

Teks: "kenapa harga bawang mahal terus sih, ga ada hubungannya sama urusan luar negeri"
Jawaban: {"themes": ["lainnya"], "primary_theme": "lainnya", "confidence": 0.9, "lokasi": [], "instansi": [], "tokoh": []}
"""

def build_combined_prompt(batch_texts):
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(batch_texts))
    return f"""Kamu classifier tema + extractor entitas cuitan bahasa Indonesia.

Daftar tema:
{THEME_LIST_STR}

Kategori entitas:
- lokasi: nama negara/kota/tempat (misal "Rusia", "Vladivostok", "Kremlin")
- instansi: nama lembaga/organisasi/acara resmi (misal "Kementerian Luar Negeri", "Eastern Economic Forum")
- tokoh: nama orang yang disebut (pejabat, tokoh publik, dll)

{FEWSHOT_COMBINED_EXAMPLES}

Untuk SETIAP teks: tentukan tema (boleh lebih dari 1, tentukan primary_theme
yang paling dominan) DAN ekstrak lokasi + instansi + tokoh sekaligus.
Kalau tidak ada entitas di kategori tertentu, gunakan array kosong — JANGAN mengarang entitas.

Teks:
{numbered}

Jawab HANYA dengan JSON array, tanpa penjelasan, tanpa markdown code block:
[
  {{"index": 1, "themes": ["kunjungan_prabowo"], "primary_theme": "kunjungan_prabowo",
    "confidence": 0.9, "lokasi": [], "instansi": [], "tokoh": ["Prabowo"]}}
]
Jumlah item HARUS sama persis dengan jumlah teks ({len(batch_texts)} item)."""


def parse_json_safe(raw_text):
    """Parse JSON, abaikan trailing garbage setelah array/object pertama yang valid.
    Fix untuk kasus 'Extra data: line X column Y' dari response Gemini."""
    decoder = json.JSONDecoder()
    raw_text = raw_text.strip()
    obj, _ = decoder.raw_decode(raw_text)
    return obj


# --- Quota tracker manual (akun gratisan) ---
api_call_count = 0

def classify_and_extract_batch(batch_texts, retry=6):
    global api_call_count
    prompt = build_combined_prompt(batch_texts)
    for attempt in range(retry):
        try:
            api_call_count += 1
            resp = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    max_output_tokens=8000,
                    **GEN_CONFIG_DETERMINISTIC,
                ),
            )
            parsed = parse_json_safe(resp.text)
            if len(parsed) != len(batch_texts):
                raise ValueError(f"Jumlah hasil ({len(parsed)}) != input ({len(batch_texts)})")

            for item in parsed:
                item["lokasi"] = canonicalize_list(item.get("lokasi", []))
                item["instansi"] = canonicalize_list(item.get("instansi", []))
                item["tokoh"] = canonicalize_list(item.get("tokoh", []))
            return parsed, True

        except Exception as e:
            err = str(e)
            if "RESOURCE_EXHAUSTED" in err or "429" in err:
                raise  # limit harian, stop total, JANGAN retry
            wait = 20 * (attempt + 1) if ("503" in err or "UNAVAILABLE" in err) else 5 * (attempt + 1)
            print(f"[WARN] gagal ({attempt+1}/{retry}): {e} -> tunggu {wait}s")
            time.sleep(wait)

    if len(batch_texts) > 8:
        print(f"[RECOVER] Split batch {len(batch_texts)} jadi 2, coba ulang lebih kecil")
        mid = len(batch_texts) // 2
        left, left_ok = classify_and_extract_batch(batch_texts[:mid], retry=3)
        right, right_ok = classify_and_extract_batch(batch_texts[mid:], retry=3)
        for r in right:
            r["index"] += mid
        return left + right, (left_ok and right_ok)

    print(f"[ERROR] Batch gagal total (bukan quota), dicap 'lainnya'")
    fallback = [{"index": i + 1, "themes": ["lainnya"], "primary_theme": "lainnya",
                 "confidence": 0.0, "lokasi": [], "instansi": [], "tokoh": []}
                for i in range(len(batch_texts))]
    return fallback, False


drive.mount('/content/drive')

df_clean["theme"] = None
df_clean["all_themes"] = None
df_clean["theme_confidence"] = None
df_clean["entities_lokasi"] = None
df_clean["entities_instansi"] = None
df_clean["entities_tokoh"] = None
df_clean["ner_failed"] = False
print("[FRESH] Mulai proses baru, tidak load checkpoint lama.")

# tahap 1: keyword prefilter
mask_pending = df_clean["theme"].isna()
keyword_signal = df_clean.loc[mask_pending, "text_clean"].apply(keyword_match_theme)

for idx, sig in keyword_signal.items():
    if sig is not None:
        df_clean.at[idx, "theme"] = sig
        df_clean.at[idx, "all_themes"] = [sig]
        df_clean.at[idx, "theme_confidence"] = 1.0
        df_clean.at[idx, "entities_lokasi"] = []
        df_clean.at[idx, "entities_instansi"] = []
        df_clean.at[idx, "entities_tokoh"] = []
        df_clean.at[idx, "ner_failed"] = False

need_llm_idx = df_clean.index[df_clean["theme"].isna()].tolist()
print(f"[INFO] {len(df_clean) - len(need_llm_idx)} baris beres (keyword prefilter), {len(need_llm_idx)} sisa ke LLM")

# tahap 2: sisanya -> LLM, satu call buat tema+lokasi+instansi+tokoh
n_batches = -(-len(need_llm_idx) // BATCH_SIZE_CLASSIFY)
print(f"[INFO] Estimasi {n_batches} request ke Gemini (~{n_batches * SLEEP_BETWEEN_CALLS / 60:.1f} menit kalau lancar tanpa retry)")
quota_hit = False

for b in tqdm(range(n_batches), desc="Classify+NER"):
    idx_batch = need_llm_idx[b * BATCH_SIZE_CLASSIFY:(b + 1) * BATCH_SIZE_CLASSIFY]
    text_batch = [df_clean.loc[i, "text_clean"] for i in idx_batch]

    try:
        results, success = classify_and_extract_batch(text_batch)
    except Exception as e:
        if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
            print(f"[STOP] Limit harian abis di batch {b}/{n_batches}. Total API call sesi ini: {api_call_count}")
            quota_hit = True
            break
        raise

    for r, df_idx in zip(results, idx_batch):
        themes_valid = [t for t in r.get("themes", ["lainnya"]) if t in THEMES] or ["lainnya"]
        primary = r.get("primary_theme")
        df_clean.at[df_idx, "theme"] = primary if primary in THEMES else themes_valid[0]
        df_clean.at[df_idx, "all_themes"] = themes_valid
        df_clean.at[df_idx, "theme_confidence"] = r.get("confidence", 0.5)
        df_clean.at[df_idx, "entities_lokasi"] = r.get("lokasi", [])
        df_clean.at[df_idx, "entities_instansi"] = r.get("instansi", [])
        df_clean.at[df_idx, "entities_tokoh"] = r.get("tokoh", [])
        df_clean.at[df_idx, "ner_failed"] = not success

    time.sleep(SLEEP_BETWEEN_CALLS)

if not quota_hit:
    print(f"\n[DONE] Semua {len(df_clean)} baris beres diklasifikasi + NER. Total API call: {api_call_count}")

df_clean["needs_review"] = df_clean["theme_confidence"] < THEME_CONFIDENCE_THRESHOLD
print("\nDistribusi tema:")
display(df_clean["theme"].value_counts())

n_failed = df_clean["ner_failed"].sum()
if n_failed > 0:
    print(f"[WARN] {n_failed} baris gagal NER setelah semua percobaan, entitasnya kosong bukan berarti tidak ada")

print("\nContoh hasil klasifikasi + NER:")
display(df_clean.loc[df_clean["theme"] != "lainnya",
        ["text_clean", "theme", "entities_lokasi", "entities_instansi", "entities_tokoh"]].head(10))

SAVE_PATH_AFTER_THEME = "/content/drive/MyDrive/df_clean_after_theme.pkl"
df_clean.to_pickle(SAVE_PATH_AFTER_THEME)
print(f"[OK] df_clean tersimpan ke {SAVE_PATH_AFTER_THEME}, {len(df_clean)} baris")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[FRESH] Mulai proses baru, tidak load checkpoint lama.
[INFO] 12722 baris beres (keyword prefilter), 4863 sisa ke LLM


Classify+NER:   0%|          | 0/122 [00:00<?, ?it/s]

[WARN] gagal (1/6): Extra data: line 42 column 2 (char 7008) -> tunggu 5s
[WARN] gagal (2/6): Extra data: line 42 column 2 (char 7008) -> tunggu 10s
[WARN] gagal (3/6): Extra data: line 42 column 2 (char 7008) -> tunggu 15s
[WARN] gagal (4/6): Extra data: line 42 column 2 (char 7008) -> tunggu 20s
[WARN] gagal (5/6): Extra data: line 42 column 2 (char 7008) -> tunggu 25s
[WARN] gagal (6/6): Extra data: line 42 column 2 (char 7008) -> tunggu 30s
[RECOVER] Split batch 40 jadi 2, coba ulang lebih kecil
[WARN] gagal (1/6): Extra data: line 43 column 1 (char 6086) -> tunggu 5s
[WARN] gagal (2/6): Extra data: line 43 column 1 (char 6086) -> tunggu 10s
[WARN] gagal (3/6): Extra data: line 43 column 1 (char 6086) -> tunggu 15s
[WARN] gagal (4/6): Extra data: line 43 column 1 (char 6086) -> tunggu 20s
[WARN] gagal (5/6): Extra data: line 43 column 1 (char 6086) -> tunggu 25s
[WARN] gagal (6/6): Extra data: line 43 column 1 (char 6086) -> tunggu 30s
[RECOVER] Split batch 40 jadi 2, coba ulang le

,count
theme,
lainnya,15638
kunjungan_prabowo,1649
rusia_putin,298



Contoh hasil klasifikasi + NER:


,text_clean,theme,entities_lokasi,entities_instansi,entities_tokoh
4,RT Kapolri Listyo Sigit Prabowo sambut kedatan...,kunjungan_prabowo,[],[],[]
22,"RT Putin bilang ke Prabowo ""Anda mengerti?"" Pu...",kunjungan_prabowo,[],[],[]
29,RT Momen ketika Putin menasehati presiden Prab...,kunjungan_prabowo,[],[],[]
48,"RT Prabowo mah galaknya sama WNI doang, ya kan...",kunjungan_prabowo,[],[],[]
58,RT Momen Presiden Prabowo Ditegur Atau Dinaseh...,kunjungan_prabowo,[],[],[]
65,Luar biasa genius. Sebuah terobosan teori ekon...,kunjungan_prabowo,[],[],[]
139,RT Nasehat Putin buat Prabowo //t.co/J46v6T3pzn,kunjungan_prabowo,[],[],[]
140,RT prabowo ke rusia karena jadi tamu kehormata...,kunjungan_prabowo,[],[],[]
141,RT Simpulkan sendiri Apakah Putin resfect deng...,kunjungan_prabowo,[],[],[]
145,Prabowo dorong investasi Rusia senilai 726 jut...,kunjungan_prabowo,[],[],[]


[OK] df_clean tersimpan ke /content/drive/MyDrive/df_clean_after_theme.pkl, 17585 baris


# Step 5 - RANKING TOP AUTHOR PER TEMA

Hanya akun non-media (sudah difilter Step 5). Skor = post count + engagement + followers.

In [30]:
df_topic = df_clean[df_clean["theme"] != "lainnya"].copy()
print(f"[INFO] {len(df_topic)} baris tersisa setelah filter tema relevan")

for col in [COL_FAVOURITED, COL_RETWEETED]:
    if col in df_topic.columns:
        df_topic[col] = pd.to_numeric(df_topic[col], errors="coerce").fillna(0)
    else:
        df_topic[col] = 0

if COL_FOLLOWERS in df_topic.columns:
    df_topic[COL_FOLLOWERS] = pd.to_numeric(df_topic[COL_FOLLOWERS], errors="coerce").fillna(0)
else:
    df_topic[COL_FOLLOWERS] = 0

df_topic["engagement"] = df_topic[COL_FAVOURITED] + df_topic[COL_RETWEETED]

top_authors_per_theme = {}

for theme in THEMES:
    if theme == "lainnya":
        continue
    sub = df_topic[df_topic["theme"] == theme]
    if sub.empty:
        print(f"[INFO] Tidak ada data untuk tema '{theme}'")
        continue

    agg = (
        sub.groupby(COL_AUTHOR_ID)
        .agg(
            jumlah_post=(COL_AUTHOR_ID, "count"),
            total_engagement=("engagement", "sum"),
            followers=(COL_FOLLOWERS, "max"),
        )
        .reset_index()
        .rename(columns={COL_AUTHOR_ID: "author_id"})
    )

    max_post = agg["jumlah_post"].max() or 1
    max_eng = agg["total_engagement"].max() or 1
    max_followers = agg["followers"].max() or 1
    agg["score"] = (
        WEIGHT_POST_COUNT * (agg["jumlah_post"] / max_post)
        + WEIGHT_ENGAGEMENT * (agg["total_engagement"] / max_eng)
        + WEIGHT_FOLLOWERS * (agg["followers"] / max_followers)
    )

    agg = agg.sort_values("score", ascending=False).head(TOP_N_AUTHORS_PER_THEME)
    agg.insert(0, "theme", theme)
    agg["rank"] = range(1, len(agg) + 1)
    top_authors_per_theme[theme] = agg

    print(f"\n=== Top Author: {theme} ===")
    display(agg[["rank", "author_id", "jumlah_post", "total_engagement", "followers", "score"]])

df_top_authors = (
    pd.concat(top_authors_per_theme.values(), ignore_index=True)
    if top_authors_per_theme else
    pd.DataFrame(columns=["theme", "author_id", "jumlah_post", "total_engagement", "followers", "score", "rank"])
)


[INFO] 1947 baris tersisa setelah filter tema relevan

=== Top Author: kunjungan_prabowo ===


,rank,author_id,jumlah_post,total_engagement,followers,score
75,1,@FREEPHONEFREES1,40,14,1568,0.600028
150,2,@LambeSahamjja,20,5878,81356,0.302168
1028,3,massamalaka,1,1800000,0,0.265000
310,4,@anggaa649001,17,0,0,0.255000
253,5,@Spt_Nusantara,14,72,742,0.210022
92,6,@Gerindra,11,158,721022,0.176997
1035,7,merdekacom,2,754300,2300000,0.172963
419,8,@grok,1,0,9031741,0.165000
162,9,@MartiPlyer,9,0,0,0.135000
582,10,@rissaa78863,8,0,7,0.120000



=== Top Author: rusia_putin ===


,rank,author_id,jumlah_post,total_engagement,followers,score
11,1,@Menlu_RI,2,63,128473,0.626091
4,2,@AndrewHolmes01,3,4,1153,0.613180
58,3,@rinjse,3,0,43,0.600007
43,4,@mfa_russia,1,77,894435,0.600000
13,5,@Ndons_Back,2,40,50640,0.538363
39,6,@kangdede78,2,11,133017,0.458022
54,7,@rang_simabua,2,6,24325,0.423560
7,8,@Deka_Ajaa,2,1,15520,0.405850
1,9,@Afzon_Dakka,2,0,4399,0.400738
47,10,@msaid_didu,1,4,850720,0.355656


# Step 6 - RINGKASAN & SENTIMEN PER TOP AUTHOR (LLM)



In [31]:
SENTIMENT_OPTIONS = ["positif", "negatif", "kontroversi"]

def build_summary_prompt(author_label, theme, texts, tool_sentiment_note=""):
    joined = "\n".join(f"- {t}" for t in texts)
    context_note = ""
    if tool_sentiment_note:
        context_note = (
            f"\nSebagai referensi tambahan (bukan patokan mutlak), tool monitoring sosial media "
            f"sebelumnya sudah memberi label sentimen per-post untuk akun ini dengan distribusi: "
            f"{tool_sentiment_note}. Gunakan ini hanya sebagai bahan pertimbangan, "
            f"keputusan akhir tetap berdasarkan isi teks yang kamu baca sendiri.\n"
        )
    return f"""Berikut kumpulan cuitan dari akun "{author_label}" tentang topik "{theme}":

{joined}
{context_note}
Tugas kamu:
1. Ringkas dalam 2-3 kalimat apa pandangan/narasi utama yang disampaikan akun ini soal topik tersebut.
2. Tentukan sentimen KESELURUHAN akun ini terhadap topik, pilih SATU dari: {", ".join(SENTIMENT_OPTIONS)}.
   - "positif" dipakai jika akun mendukung/memuji topik ini.
   - "negatif" dipakai jika akun mengkritik/menentang topik ini.
   - "kontroversi" dipakai jika pendapat akun ini memicu perdebatan/pro-kontra, menyampaikan klaim yang kontroversial, atau mencampur pujian dan kritik sekaligus.
3. Berikan alasan singkat (1 kalimat) untuk sentimen tersebut.

Jawab HANYA dengan JSON, tanpa penjelasan tambahan, tanpa markdown code block:
{{"summary": "...", "sentiment": "...", "reason": "..."}}"""


def summarize_author(author_label, theme, texts, tool_sentiment_note="", retry=3):
    global api_call_count
    prompt = build_summary_prompt(author_label, theme, texts, tool_sentiment_note)
    for attempt in range(retry):
        try:
            api_call_count += 1
            resp = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    max_output_tokens=2000,
                    **GEN_CONFIG_DETERMINISTIC,
                ),
            )
            parsed = parse_json_safe(resp.text)
            if parsed.get("sentiment") not in SENTIMENT_OPTIONS:
                parsed["sentiment"] = "kontroversi"
            return parsed
        except Exception as e:
            err = str(e)
            if "RESOURCE_EXHAUSTED" in err or "429" in err:
                print(f"[STOP] Limit harian abis pas summarize @{author_label}. Total API call sesi ini: {api_call_count}")
                raise  # ga usah retry, quota emang abis
            wait = 5 * (attempt + 1)
            print(f"[WARN] gagal summarize {author_label} (percobaan {attempt+1}/{retry}): {e} -> tunggu {wait}s")
            time.sleep(wait)
    return {"summary": "(gagal diringkas otomatis)", "sentiment": "kontroversi", "reason": "error API"}

print("Fungsi summarization siap.")

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

results = []
quota_hit_summary = False

for _, row in df_top_authors.iterrows():
    author_id = row["author_id"]
    theme = row["theme"]

    author_posts_df = df_topic[(df_topic[COL_AUTHOR_ID] == author_id) & (df_topic["theme"] == theme)]
    posts = author_posts_df["text_clean"].dropna().tolist()

    tool_sentiment_note = ""
    if COL_SENTIMENT in author_posts_df.columns:
        counts = author_posts_df[COL_SENTIMENT].dropna().value_counts()
        if not counts.empty:
            total = counts.sum()
            tool_sentiment_note = ", ".join(f"{label} {round(100 * n / total)}%" for label, n in counts.items())

    posts = posts[:MAX_POSTS_PER_AUTHOR_SUMMARY]
    if not posts:
        continue

    print(f"Meringkas @{author_id} | tema={theme} | {len(posts)} post ...")
    try:
        result = summarize_author(author_id, theme, posts, tool_sentiment_note)
    except Exception as e:
        if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
            quota_hit_summary = True
            break
        raise

    results.append({
        "theme": theme,
        "rank": int(row["rank"]),
        "author_id": author_id,
        "score": round(row["score"], 2),
        "summary": result.get("summary", ""),
        "sentiment": result.get("sentiment", "kontroversi"),
        "reason": result.get("reason", ""),
    })

df_summary = pd.DataFrame(results)

if quota_hit_summary:
    print(f"\n[STOP] Berhenti di tengah karena quota habis. {len(results)} author sudah beres, sisanya belum.")
else:
    print(f"\n[DONE] Semua author beres diringkas. Total API call sesi ini: {api_call_count}")

print("\n=== Hasil akhir: ringkasan & sentimen per top author ===")
display(df_summary)

Fungsi summarization siap.
Meringkas @@FREEPHONEFREES1 | tema=kunjungan_prabowo | 15 post ...
Meringkas @@LambeSahamjja | tema=kunjungan_prabowo | 15 post ...
Meringkas @massamalaka | tema=kunjungan_prabowo | 1 post ...
Meringkas @@anggaa649001 | tema=kunjungan_prabowo | 15 post ...
Meringkas @@Spt_Nusantara | tema=kunjungan_prabowo | 14 post ...
Meringkas @@Gerindra | tema=kunjungan_prabowo | 11 post ...
Meringkas @merdekacom | tema=kunjungan_prabowo | 2 post ...
Meringkas @@grok | tema=kunjungan_prabowo | 1 post ...
Meringkas @@MartiPlyer | tema=kunjungan_prabowo | 9 post ...
Meringkas @@rissaa78863 | tema=kunjungan_prabowo | 8 post ...
Meringkas @@Menlu_RI | tema=rusia_putin | 2 post ...
Meringkas @@AndrewHolmes01 | tema=rusia_putin | 3 post ...
Meringkas @@rinjse | tema=rusia_putin | 3 post ...
Meringkas @@mfa_russia | tema=rusia_putin | 1 post ...
Meringkas @@Ndons_Back | tema=rusia_putin | 2 post ...
Meringkas @@kangdede78 | tema=rusia_putin | 2 post ...
Meringkas @@rang_simabua 

,theme,rank,author_id,score,summary,sentiment,reason
0,kunjungan_prabowo,1,@FREEPHONEFREES1,0.60,Akun ini menarasikan bahwa kunjungan Prabowo Subianto dan militer Indonesia merupakan bagian dari aliansi strategis untuk bekerja sama dengan FBI dan Departemen Kehakiman AS dalam mengungkap kejahatan internasional. Narasi tersebut juga mengaitkan kunjungan ini dengan upaya pemulihan aset negara yang dicuri serta pengungkapan konspirasi pembunuhan Ani Yudhoyono oleh pihak asing.,positif,Akun tersebut secara eksplisit memuji Prabowo Subianto dan militer Indonesia sebagai mitra strategis yang mendukung investigasi kriminal internasional dan aliansi kebebasan yang mereka bentuk.
1,kunjungan_prabowo,2,@LambeSahamjja,0.30,"Akun ini menyoroti kunjungan Prabowo ke Rusia dengan memaparkan detail pidato dan agenda kerja sama bilateral, namun di saat yang sama juga mengangkat narasi kritis terkait isu keracunan program makan siang gratis yang diberitakan media Rusia. Narasi yang dibangun mencakup pembelaan terhadap citra Prabowo di tengah kritik publik serta laporan mengenai tindak lanjut penertiban kawasan hutan setelah kunjungan tersebut.",kontroversi,Akun ini menyajikan campuran antara informasi resmi yang mendukung agenda pemerintah dengan sorotan tajam terhadap kegagalan program domestik yang memicu perdebatan di media internasional.
2,kunjungan_prabowo,3,massamalaka,0.27,"Akun tersebut menyoroti momen keakraban dan ekspresi emosional para pejabat Indonesia, termasuk Bahlil Lahadalia dan Teddy Indra Wijaya, saat mendampingi Prabowo Subianto dalam kunjungan luar negeri. Narasi ini menekankan pada suasana positif dan antusiasme para tokoh yang terlibat dalam delegasi tersebut.",positif,Akun tersebut menggambarkan momen kunjungan dengan nada yang ceria dan apresiatif terhadap ekspresi para pejabat yang mendampingi Prabowo.
3,kunjungan_prabowo,4,@anggaa649001,0.26,"Akun ini menyoroti kunjungan Presiden Prabowo ke Vladivostok, Rusia, sebagai tamu kehormatan utama dalam forum EEF untuk mempererat hubungan bilateral kedua negara. Narasi yang dibangun menekankan pada diplomasi aktif Prabowo dan apresiasi atas kehadirannya dalam memenuhi undangan Presiden Vladimir Putin.",positif,Seluruh cuitan bernada mendukung dan memuji langkah diplomasi Prabowo dengan menggunakan tagar apresiatif seperti #JagaIndonesia dan #TerimakasihPakBowo.
4,kunjungan_prabowo,5,@Spt_Nusantara,0.21,Akun ini menyoroti kunjungan Presiden Prabowo Subianto ke Eastern Economic Forum (EEF) di Vladivostok sebagai langkah strategis untuk mempererat hubungan bilateral dan kerja sama ekonomi antara Indonesia dan Rusia. Narasi yang dibangun menekankan posisi Indonesia sebagai mitra utama Rusia serta komitmen kedua pemimpin dalam meningkatkan investasi dan kolaborasi di berbagai sektor.,positif,"Seluruh konten secara konsisten mempromosikan narasi kedekatan diplomatik, kerja sama ekonomi yang saling menguntungkan, dan apresiasi terhadap peran strategis Presiden Prabowo di kancah internasional."
5,kunjungan_prabowo,6,@Gerindra,0.18,"Akun @Gerindra menyoroti kunjungan Presiden Prabowo ke Rusia sebagai langkah strategis untuk mempererat hubungan bilateral melalui kerja sama investasi konkret. Fokus utama kunjungan ini adalah mendorong kolaborasi antara Danantara dan RDIF dalam membiayai proyek strategis di berbagai sektor seperti energi, pangan, dan industri hilir.",positif,Narasi akun secara konsisten memuji inisiatif Presiden Prabowo dalam membangun kemitraan ekonomi yang pragmatis dan bermanfaat bagi kedua negara.
6,kunjungan_prabowo,7,merdekacom,0.17,"Presiden Prabowo Subianto melakukan kunjungan kerja ke Vladivostok, Rusia, untuk menghadiri Forum Ekonomi Timur (EEF) ke-11. Kunjungan ini menyoroti hubungan strategis antara Indonesia dan Rusia, di mana Presiden Putin memberikan penghormatan khusus kepada Prabowo sebagai mitra kunci di kawasan Asia Pasifik.",positif,Narasi yang disampaikan menonjolkan pengakuan internasional terhadap posisi Indonesia dan upaya penguatan kerja sama eko